In [1]:
import gc
# import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [2]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

current_baselines = ["emo", "emo_v2", "ddpo", "b2diffurl"] 
baselines = ["md3po", "emo", "emo_v2", "ddpo", "b2diffurl", "dpok"]
eval_path = Path("outputs/")
training_data_eval_file = "training_metrics.json"
evaluation_data_eval_file = "eval_metrics.json"

def get_training_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    all_metric_entries = []
    for entry in list_of_metrics:
        all_metric_entries.append(entry['metrics'])
    df = pd.DataFrame(all_metric_entries)
    return df

def get_eval_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    
    scores = ['bert_reward', 'clip_reward']
    eval_summary = list()
    for entry in list_of_metrics:
        for score in scores:
            eval_summary.append({
                "value": round(np.array(entry[score]).mean(), 4),
                "score": score,
                "epoch": entry['epoch']
            })
    eval_df = pd.DataFrame(eval_summary)
    return eval_df
    
def get_metrics():
    training_eval_df = pd.DataFrame()
    eval_df = pd.DataFrame()
    for baseline in baselines:
        baseline_path = eval_path / baseline
        training_data_eval_file_path = baseline_path / "training_evals" / training_data_eval_file if baseline not in current_baselines else baseline_path / "seed_123/training_evals" / training_data_eval_file
        eval_data_file_path = baseline_path / "evals" / evaluation_data_eval_file if baseline not in current_baselines else baseline_path / "seed_123/evals" / evaluation_data_eval_file

        
        df = get_training_metrics_df(training_data_eval_file_path)
        df["method"] = baseline
        training_eval_df = pd.concat([training_eval_df, df], ignore_index=True)

        df = get_eval_metrics_df(eval_data_file_path)
        df["method"] = baseline
        eval_df = pd.concat([eval_df, df], ignore_index=True)
    return training_eval_df, eval_df

In [21]:
training_eval_df, eval_df = get_metrics()

import matplotlib.pyplot as plt

temp_eval_df = eval_df[eval_df['score'] == 'clip_reward']
temp_eval_df = temp_eval_df[temp_eval_df['method'].isin(list(set(baselines)-set(["dpok_old"])))]
eval_summary = temp_eval_df.pivot_table(
    index=['method', 'score'],
    columns='epoch',
    values='value',
)
# eval_summary.T.iloc[:, 0].plot()
# plt.title("Bert Reward")
# plt.show()
# eval_summary.T.iloc[:, 1].plot()
# plt.title("Clip Reward")
# plt.show()

transposed_eval_summary = eval_summary.T
transposed_eval_summary = transposed_eval_summary.reset_index().droplevel(level=1, axis=1)
# transposed_eval_summary[transposed_eval_summary.index.isin([2]+list(range(0, 50, 5)))].rolling(window=2).mean().round(3).plot(marker='o')
transposed_eval_summary['queries'] = transposed_eval_summary['epoch'].astype(int)*256
transposed_eval_summary.pivot_table(
    index='queries',
    values=['b2diffurl', 'ddpo', 'emo', 'emo_v2'],
)#.rolling(window=1).mean().plot(marker='*') #.iloc[0::3]

method,b2diffurl,ddpo,emo,emo_v2
queries,,,,
512,0.33600,0.3429,0.3384,0.3258
1024,0.34230,0.3412,0.3281,0.3352
1536,0.33760,0.3428,0.3311,0.3301
2048,0.33290,0.3404,0.3353,0.3513
2560,0.34600,0.3451,0.3340,0.3533
3072,0.34510,0.3408,0.3368,0.3491
3584,0.34440,0.3420,0.3336,0.3350
4096,0.34580,0.3448,0.3364,0.3536
4608,0.34430,0.3469,0.3405,0.3521


In [19]:
cols = ['advantage_mean', 'approx_kl', 'current_reward_mean', 'current_reward_std', 'epoch', 'loss', 'parameter_update_norm_mean',
    'raw_reward_mean', 'raw_reward_std', 'replay_reward_mean',
    'replay_reward_std', 'reward_mean', 'reward_std', 'selected_samples', 'skipped_updates','method']
training_eval_df = training_eval_df[cols]
_training_eval_df = training_eval_df[training_eval_df['method'].isin(['ddpo', 'emo_v2'])]
_training_eval_df[cols].pivot_table(
    index=['epoch'],
    columns=['method'],
    values=['reward_mean'], #'loss', 
    # aggfunc='mean'
).round(3)#.rolling(window=1).mean().plot(kind='line') ##

reward_mean       
method        ddpo emo_v2
epoch                    
1            0.324  0.208
2            0.325  0.203
3            0.325  0.206
4            0.332  0.208
5            0.335  0.207
6            0.334  0.208
7            0.334  0.210
8            0.335  0.209
9            0.330  0.206
10           0.339  0.209
11           0.335  0.205
12           0.338  0.208
13           0.331  0.204
14           0.339  0.206
15           0.335  0.204
16           0.340  0.210
17           0.331  0.205
18           0.338  0.208
19           0.337  0.206
20           0.342  0.205
21           0.341  0.205
22           0.341  0.210
23           0.344  0.207
24           0.344  0.203
25           0.344  0.207
26           0.342  0.207
27           0.346  0.210
28           0.350  0.215
29           0.347  0.212
30           0.341  0.208
31           0.348  0.205
32           0.345  0.207
33           0.351  0.206
34           0.344  0.203
35           0.349  0.208
36           0.350  0.206
37           0.354  0.206
38           0.353  0.205
39           0.353  0.202
40           0.350  0.200
41           0.349  0.199
42           0.357  0.204
43           0.356  0.201
44           0.355  0.192
45           0.357  0.177
46           0.353  0.101
47           0.350    NaN
48           0.355    NaN
49           0.360    NaN
50           0.355    NaN